# Dataset Coral Sol - Segmentação de Imagens - Análise Exploratória de Dados (EDA)

**Autor:** Matheus Santos  
**Data:** 19-02-2026 

## Objetivo
Realizar uma análise exploratória completa de um conjunto de dados de segmentação de imagens contendo:
- `images/`: imagens de treinamento e teste
- `labels/`: pontos de segmentação (polígonos) para cada imagem

In [ ]:
!pip install opencv-python nbconvert

## Imports necessários

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import math
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt

import cv2


## Contantes

In [3]:
SUPPORTED_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SUPPORTED_LABEL_EXTENSIONS = {".txt"}

DEFAULT_RANDOM_SEED = 42
DEFAULT_OVERLAY_SAMPLES = 12
DEFAULT_DUPLICATE_GROUP_MIN_SIZE = 2

MATCH_SIMILARITY_THRESHOLD = 0.35
ENABLE_FALLBACK_ORDER_MATCHING = True

## Funções

In [4]:
@dataclass(frozen=True)
class ImageItem:
    """Represents a single image file and its split metadata.

    Attributes:
        path: Full path to the image file.
        split: Dataset split inferred from the directory name (e.g., 'train', 'test').
        width: Image width in pixels.
        height: Image height in pixels.
        suffix: File extension (lowercase).
    """
    path: Path
    split: str
    width: int
    height: int
    suffix: str


@dataclass(frozen=True)
class LabelItem:
    """Represents a single label file and its split metadata.

    Attributes:
        path: Full path to the label file.
        split: Dataset split inferred from the directory name (e.g., 'train', 'test').
    """
    path: Path
    split: str


# =========================
# Utility functions
# =========================

def set_global_seed(seed: int = DEFAULT_RANDOM_SEED) -> None:
    """Set random seeds for reproducibility.

    Args:
        seed: Seed value used by Python's random module and NumPy.

    Returns:
        None.
    """
    random.seed(seed)
    np.random.seed(seed)


def infer_split(path: Path) -> str:
    """Infer dataset split from a file path.

    The function searches parent directories for common split names
    such as 'train', 'test', or 'val'. If none is found, it returns 'unknown'.

    Args:
        path: File path.

    Returns:
        Split name as a lowercase string.
    """
    parts_lower = [p.lower() for p in path.parts]
    for split_name in ("train", "test", "val", "valid", "validation"):
        if split_name in parts_lower:
            return "val" if split_name in {"val", "valid", "validation"} else split_name
    return "unknown"


def list_files(root_dir: Path, extensions: Sequence[str]) -> List[Path]:
    """List files under a root directory matching given extensions.

    Args:
        root_dir: Root directory to recursively scan.
        extensions: Allowed file extensions (lowercase), including the dot.

    Returns:
        List of matching file paths.
    """
    if not root_dir.exists():
        raise FileNotFoundError(f"Directory not found: {root_dir}")

    ext_set = {e.lower() for e in extensions}

    # Recursively search for files with matching extensions
    return [
        p for p in root_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in ext_set
    ]


def load_images(images_dir: Path) -> List[ImageItem]:
    """Load image metadata (path, split, width, height).

    Args:
        images_dir: Directory containing images.

    Returns:
        List of ImageItem objects.
    """
    image_paths = list_files(images_dir, sorted(SUPPORTED_IMAGE_EXTENSIONS))
    items: List[ImageItem] = []

    for path in image_paths:
        try:
            # Open image to retrieve its dimensions
            with Image.open(path) as img:
                width, height = img.size
        except Exception as exc:
            warnings.warn(f"Failed to read image: {path} | {exc}")
            continue

        items.append(
            ImageItem(
                path=path,
                split=infer_split(path),
                width=int(width),
                height=int(height),
                suffix=path.suffix.lower(),
            )
        )

    return items


def load_labels(labels_dir: Path) -> List[LabelItem]:
    """Load label file paths and split metadata.

    Args:
        labels_dir: Directory containing label files.

    Returns:
        List of LabelItem objects.
    """
    label_paths = list_files(labels_dir, sorted(SUPPORTED_LABEL_EXTENSIONS))
    return [LabelItem(path=p, split=infer_split(p)) for p in label_paths]


def draw_polygon_on_image(img: np.ndarray, annotation_line: str) -> np.ndarray:
    """Draw a YOLO-seg polygon on an image.

    Expected format:
        <class_id> x1 y1 x2 y2 ... xn yn

    Coordinates (x, y) must be normalized in [0, 1].
    The polygon is converted to pixel coordinates using image dimensions.

    Args:
        img: Image as a NumPy array (H, W, C) in BGR (OpenCV default) or RGB.
        annotation_line: One line from a .txt annotation file.

    Returns:
        A copy of the image with the polygon drawn.
    """
    parts = annotation_line.strip().split()

    # A valid polygon must contain at least class_id + 3 points (6 coords)
    if len(parts) < 7:
        return img

    try:
        coords = list(map(float, parts[1:]))

        # Coordinates must be in pairs (x, y)
        if len(coords) % 2 != 0:
            return img

        class_id = int(float(parts[0]))
    except ValueError:
        return img

    h, w = img.shape[:2]
    points = []

    # Convert normalized coordinates to pixel coordinates
    for i in range(0, len(coords), 2):
        x_px = int(coords[i] * w)
        y_px = int(coords[i + 1] * h)
        points.append([x_px, y_px])

    points_arr = np.array(points, dtype=np.int32)

    out = img.copy()

    # Draw polygon outline
    cv2.polylines(
        out,
        [points_arr],
        isClosed=True,
        color=(0, 255, 0),
        thickness=2,
    )

    # Optionally write class_id near the first vertex
    cv2.putText(
        out,
        f"cls={class_id}",
        (int(points_arr[0, 0]), int(points_arr[0, 1])),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )

    return out


def find_label_for_image(image_path: Path, labels_dir: Path) -> Path | None:
    """Find the corresponding label file for an image.

    Default rule: same base filename (stem) with .txt extension,
    searched recursively inside labels_dir.

    Example:
        images/train/abc.jpg -> labels/train/abc.txt

    Args:
        image_path: Path to the image file.
        labels_dir: Root directory of label files.

    Returns:
        Path to the label file if found, otherwise None.
    """
    target_name = f"{image_path.stem}.txt"
    candidates = list(labels_dir.rglob(target_name))
    return candidates[0] if candidates else None


def show_images_with_polygons(
    image_paths: list[Path],
    labels_dir: Path,
    max_polygons_per_image: int = 50,
) -> None:
    """Display images with polygons drawn from label files.

    Args:
        image_paths: List of image file paths.
        labels_dir: Root directory containing label files.
        max_polygons_per_image: Maximum number of polygons drawn per image.

    Returns:
        None.
    """
    for img_path in image_paths:
        label_path = find_label_for_image(img_path, labels_dir)

        if label_path is None:
            print(f"[NO LABEL] {img_path.name}")
            continue

        # OpenCV reads images in BGR format
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"[IMAGE ERROR] {img_path}")
            continue

        # Read and clean label lines
        lines = label_path.read_text(
            encoding="utf-8", errors="ignore"
        ).splitlines()
        lines = [ln.strip() for ln in lines if ln.strip()]

        if not lines:
            print(f"[EMPTY LABEL] {img_path.name}")
            continue

        out = img_bgr.copy()

        # Draw up to max_polygons_per_image polygons
        for ln in lines[:max_polygons_per_image]:
            out = draw_polygon_on_image(out, ln)

        # Convert BGR -> RGB for matplotlib display
        out_rgb = cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(12, 6))
        plt.imshow(out_rgb)
        plt.title(
            f"{img_path.name} | polygons="
            f"{min(len(lines), max_polygons_per_image)}"
        )
        plt.axis("off")
        plt.show()


## Configurações

Defina `DATASET_ROOT` para a pasta do conjunto de dados que contém `images/` e `labels/`.


In [ ]:
# =========================
# Dataset path configuration
# =========================

"""
This section initializes reproducibility settings and defines
the main dataset directories (root, images, and labels).

Expected structure:
DATASET_ROOT/
    ├── images/
    └── labels/
"""

# Set global random seed for reproducibility
set_global_seed()

# Root directory of the dataset
DATASET_ROOT = Path(
    ".\\coral_images_lote_mergulho"
)

# Subdirectories for images and labels
IMAGES_DIR = DATASET_ROOT / "images"
LABELS_DIR = DATASET_ROOT / "labels"

# Print directory information and existence checks
print("Dataset root:", DATASET_ROOT)
print("Images dir:", IMAGES_DIR, "| exists:", IMAGES_DIR.exists())
print("Labels dir:", LABELS_DIR, "| exists:", LABELS_DIR.exists())


## 1) Dataset - Visão Geral

Esta seção responde:
- Quantas imagens existem por divisão?
- Quais são os formatos de imagem?
- Quais são as resoluções e proporções típicas?


In [ ]:
# =========================
# Load images and labels, create DataFrames
# =========================

"""
This section loads images and label files from the respective directories,
creates pandas DataFrames with metadata, and performs a basic exploration
of the dataset.
"""

# Load all images and labels into lists of dataclasses
images = load_images(IMAGES_DIR)
labels = load_labels(LABELS_DIR)

# -------------------------
# Create a DataFrame for images
# -------------------------
images_df = pd.DataFrame([
    {
        "split": i.split,                   # Dataset split: train/test/val/unknown
        "image_path": str(i.path),          # Full path to the image
        "width": i.width,                   # Image width in pixels
        "height": i.height,                 # Image height in pixels
        "suffix": i.suffix,                 # File extension (e.g., .jpg)
        "aspect_ratio": i.width / max(i.height, 1),  # Width/height ratio
        "n_pixels": i.width * i.height,     # Total number of pixels
    }
    for i in images
])

# -------------------------
# Create a DataFrame for labels
# -------------------------
labels_df = pd.DataFrame([
    {
        "split": l.split,                   # Dataset split inferred from path
        "label_path": str(l.path),          # Full path to the label file
        "suffix": l.path.suffix.lower(),    # Label file extension (.txt)
    }
    for l in labels
])

# -------------------------
# Display first rows of the DataFrames
# -------------------------
display(images_df.head(400))
display(labels_df.head(400))

# -------------------------
# Print basic dataset statistics
# -------------------------
print("Images count:", len(images_df))
print("Labels count:", len(labels_df))

# Number of images per split
print("\nImages by split:")
display(
    images_df.groupby("split")
    .size()
    .rename("n_images")
    .reset_index()
)

# Number of label files per split
print("\nLabels by split:")
display(
    labels_df.groupby("split")
    .size()
    .rename("n_label_files")
    .reset_index()
)

# Distribution of image formats/extensions
print("\nImage formats:")
display(
    images_df["suffix"]
    .value_counts()
    .rename_axis("suffix")
    .reset_index(name="count")
)


In [ ]:
train_images = sorted((IMAGES_DIR / "train").rglob("*"))
train_images = [p for p in train_images if p.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS]

random.shuffle(train_images)
sample = train_images[:20]

show_images_with_polygons(sample, LABELS_DIR)


In [ ]:
!jupyter nbconvert --to html eda_coral_sol_segmentacao.ipynb